In [ ]:
## Packages Import
%matplotlib widget
import copy
import sys
import time
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

sys.path.append("./src/")
from VLMSurface import VLMSurface
from VLMSolver import VLMSolver

In [ ]:
## Utilities

def chord_fn(n, ar, b, sym, space, shape, lam=1.8):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of spanwise stations
        ar    -> aspect ratio
        b     -> wing span
        sym   -> symmetric or not
        space -> uniform or cosine spanwise spacing
        shape -> wing planform shape, can be "rectangular", "elliptical" or "tapered"
        lam   -> taper ratio, only needed for tapered shape

    Output:
        c     -> chord length distribution along spanwise direction, shape (n,)
    """
    le = (0.5 * b)
    s_min, s_max = 0.0, le
    if space :
        if sym :
            theta = np.linspace(np.pi/2, np.pi, n)
            s = le*(-np.cos(theta))
        else :
            theta = np.linspace(0, np.pi, n)
            s = le*(1-np.cos(theta))/2
    else :
        s = np.linspace(s_min, s_max, n)
    match shape:
        case "rectangular":
            c = b/ar * np.ones(n)
        case "elliptical":
            c = 4*b/(np.pi*ar) * np.sqrt(1 - (s/le)**2)
        case "tapered":
            c = - s*4*(lam - 1)/(ar*(lam + 1)) + 4*le*lam/(ar*(lam + 1))
    return c

def plot3D(surfaces, show_mirror):
    """ 
    Plot all the surface in 3D

    Input
        surfaces    -> a list of VLMSurface
        show_mirror -> enable the display of boundary conditions 
    """
    fig_3d = plt.figure(figsize=(14, 8),constrained_layout=True)
    ax4 = fig_3d.add_subplot(111, projection='3d')
    first = {"wing":True, "wake":True, "mir_wing":True, "mir_wake":True}             # to only have one legend per item
    for surface in surfaces:
        for i, panel in enumerate(surface.wing_panels["real"]):
                pnt    = copy.copy(panel.pnt)
                vrt    = copy.copy(panel.vrt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                vrt.append(vrt[0])
                vrt_plt = np.array(vrt)
                if first["wing"] :               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black', label='Wing')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8, label='Vortex rings')
                        first["wing"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8)
        for i, panel in enumerate(surface.wake_panels["real"]):
            pnt    = copy.copy(panel.pnt)
            # Close the polygon shape
            pnt.append(pnt[0])
            pnt_plt = np.array(pnt)
            if first["wake"]:               # In order to have just one legend
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6, label='Wake')
                    first["wake"] = False
            else :
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6)
        if show_mirror:
            for i, panel in enumerate(surface.wing_panels["mirror"]):
                    pnt    = copy.copy(panel.pnt)
                    vrt    = copy.copy(panel.vrt)
                    # Close the polygon shape
                    pnt.append(pnt[0])
                    pnt_plt = np.array(pnt)
                    vrt.append(vrt[0])
                    vrt_plt = np.array(vrt)
                    if first["mir_wing"] :               # In order to have just one legend
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray', label='Mirrored wing')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8, label='Mirrored vortex rings')
                            first["mir_wing"] = False
                    else :
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8)
            for i, panel in enumerate(surface.wake_panels["mirror"]):
                pnt    = copy.copy(panel.pnt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                if first["mir_wake"]:               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6, label='Mirrored wake')
                        first["mir_wake"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6)
            
    ax4.view_init(elev=30, azim=40)
    x_lim = ax4.get_xlim3d()
    y_lim = ax4.get_ylim3d()
    z_lim = ax4.get_zlim3d()
    ax4.set_xlim(x_lim[0],x_lim[1])
    ax4.set_ylim(y_lim[0],y_lim[1]) 
    ax4.set_zlim(z_lim[1],z_lim[0])
    ax4.set_box_aspect([-x_lim[0]+x_lim[1],-y_lim[0]+y_lim[1],-z_lim[0]+z_lim[1]])
    ax4.set_xlabel("y", fontstyle='italic')
    ax4.set_ylabel("x", fontstyle='italic')
    ax4.set_zlabel("z", fontstyle='italic')
    ax4.xaxis.set_major_locator(MultipleLocator(0.5))
    ax4.yaxis.set_major_locator(MultipleLocator(1))
    ax4.zaxis.set_major_locator(MultipleLocator(0.5))

    ax4.set_title('3D view')
    ax4.legend()


In [ ]:
## Parameters Definition

# Wing geometry
B = 1        # wing span                 [m]
AR = 5       # aspect ration             [-]
ALPHA  = 10  # angle of attack           [deg] - positive definite for counterclockwise rotations about y-axis
BETA   = 0   # drift angle               [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 0   # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 0    # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 
PHI    = 0    # twist tip angle           [deg] - negative for washout

SYM   = True           # symmetric wing configuration
SPACE = True          # spacing distribution, False = uniform, True = cos
SHAPE = "rectangular"   # shape of the wing
TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex
FREE  = True         # free surface enable

# Flow properties
U = 1.0     # inflow velocity [m/s]

# Wing discretization
N = 15      # number of panels in spanwise direction
M = 4      # number of panels in chordwise direction

# Time simulation parameters
T  = 2   # length of the simulation in s
DT = 0.1    # time step lentgh 




In [ ]:
## Test VLMSurface
surface1 = VLMSurface(origin=np.array([-0.3,0,1]),
                     plan=1,
                     boundary=FREE,
                     shedding="both",
                     b=1.2*B,
                     c=chord_fn((N+1), 10, 1.2*B, SYM, SPACE, "elliptical"),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(0),
                     delta=np.deg2rad(0),
                     phi=np.deg2rad(PHI),
                     sym=SYM,
                     space=SPACE,
                     n=N,
                     m=M)
surface2 = VLMSurface(origin=np.array([0.8,0,1]),
                     plan=1,
                     boundary=FREE,
                     shedding="both",
                     b=0.5*B,
                     c=chord_fn((N+1), 10, 0.5*B, SYM, SPACE, "tapered"),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(20),
                     delta=np.deg2rad(0),
                     phi=np.deg2rad(PHI),
                     sym=SYM,
                     space=SPACE,
                     n=N,
                     m=M)
surface3 = VLMSurface(origin=np.array([0,0,0]),
                     plan=0,
                     boundary=FREE,
                     shedding="right",
                     b=2*B,
                     c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(LAMBDA),
                     delta=np.deg2rad(DELTA),
                     phi=np.deg2rad(PHI),
                     sym=False,
                     space=SPACE,
                     n=N,
                     m=M)

rectangle = VLMSurface(origin=np.array([0,0,0]),
                     plan=0,
                     boundary=FREE,
                     shedding="right",
                     b=B,
                     c=chord_fn((N+1), AR, B, SYM, SPACE, "rectangular"),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(LAMBDA),
                     delta=np.deg2rad(DELTA),
                     phi=np.deg2rad(PHI),
                     sym=SYM,
                     space=SPACE,
                     n=N,
                     m=M) 

surface1._build_wing()
surface2._build_wing()
surface3._build_wing()
rectangle._build_wing()

vlm = VLMSolver([surface1, surface2, surface3], np.array([U,0,0]), FREE)

vlm._time_sim(t=T, dt=DT, distribution=TIME)
print("done")

plot3D([surface1, surface2, surface3],True)


In [ ]:
## Cl versus alpha
alpha = np.linspace(0,11,12)
timee = np.linspace(1,8,8)
cl    = []
cl_l  = []
for ti in timee :
    vlm = VLMSolver(b=B,
                c=chord_fn((2+1), AR, B, SYM, SPACE, SHAPE),
                alpha=np.deg2rad(ALPHA),
                lamb=np.deg2rad(LAMBDA),
                delta=np.deg2rad(DELTA),
                phi=np.deg2rad(PHI),
                sym=SYM,
                space=SPACE,
                u_inf=np.array([U, 0.0, 0.0]),
                n=2,m=2)
    vlm._time_sim(ti, DT, distribution=TIME)
    l = vlm._secondary_computation()[1]
    cl.append(2*l*AR/(B**2))
    l = vlm._kuttas_loads()[1]
    cl_l.append(2*l*AR/(B**2))
print(timee)
print(cl)
print(cl_l)

